# Crawl corpus ViBioMIR trên Kaggle

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; chưa có OCR hoặc browser render JavaScript.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).


## 1. Cấu hình — chỉnh ID tại đây

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên `DELAY`, `ATTEMPTS` và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus.parquet`.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL

SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RUN_TAG = None            # None: tự đặt ids_<ID_FROM>_<ID_TO>; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
DATASET_REVISION = '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7'
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/links_corpus.parquet'

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
tag = RUN_TAG or f'ids_{ID_FROM}_{ID_TO}'
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
SOURCE_PATH = WORKDIR / 'data' / 'links_corpus.parquet'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
không ép bản lock Windows của repo. Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot của code trong repo vào output; không tải code từ mạng.
Các file `.py` sinh ra có thể mở để xem. Snapshot có sẵn của run được giữ nguyên khi resume.
Muốn cập nhật snapshot sau khi sửa code repo, chạy `scripts/build_kaggle_notebook.py` để dựng lại notebook.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO5TW=f36@EA1|1hCj?6M_SmpDdCkun_HL5#$S;W$MBh1iUChvY<i^X#r9Mk3H64}B`oAO(t`2oNVIf;1?aqJ5Gc`k)~HL;WTF&dls`MUk>y)PP0q&biIGd~@co)oQ&v8bvY@OeTdGD_Km}f)rnqf`yWglQb`6kTafy_|M8D;=3Y>M97rbm*Rls6CnyVN)_WQQ2Zcbs!Y6At9AXFjI&e~jH_|RRW7by8>uvAK??SukZF=LOK~?Xp;6Tw<Bw%lKTLQ&iRAA5xrigibN%UyJm8tA*FMVA#Hz`7L2Q+ZM|iBs6n2xw^T)Y>e45wa%R~lgC_-NF>({K*=n%>qhSbE_lq;1Uc-ZOF5)OfZ*;8V>ek~LusNkVXCbfIluQB{nq9|3O4fdjxhq+$$h|Vv>Vd07d%4IShIAt;F{=#Xu=hA{S&UjF`M7SW(VUG7vDigP1y_cja=8^nZxXwFo+;uukGkSRz$--?4&L`X@js>AM1Rj<*U%O=}yoopr<yhp9Fc%PWsIItm)D;`l8Q|Sf<1$8SJ<=;~i0~<yQ7#O8KTHGPZ#P*z9)>=*Jg(F2rey)^m3Uf8CBnfIRf?+_Op<nU?47nf&BOqC-eItnmr*g;SYNjSz?H*PD@T8jq};XZP>p3uBbQ!#c(yAsPvi)Gz_JcAQLZtkdhJn+9ZM^Xk1Lv-$5|vaMdY+i^mnI)fYvs9Wl)RWv6Q}vfHSln)-mET7wlI&D#d-IpmLq}G_8&ri)9}3BA8SL7&$O?r;&PjTNg0uF=1DkwmdOS9ubDX?4eO34l~p^P-4K2)Q%%Vn9T5{RFN~($`mOxN@muyv{V6^mOZuV_--WJh7UA>GwiUDjDQ#UV8<oV9p=ace<O)xtTeaV!?stXem2#w=l;La9<5<UuCL_0h^NC;<Sa=Grd?fa<dK1b%JWJDsR}8_b_^OAr)Ng5f~q<#EGb+&j?!J%`I*h3%Dbrjl1I<+traq>H*UEadt;!DMB-XfopSVC`|!%mKQ0sKAuNhSAxIV-8~}hE&{?Z^a4+9t2VB}SB8pfr5y4&tj4JZVHv!Lu_RO%E3O>~?y*{#=SlnB~Jd!|^<PPl(atjiJoHF+gUOa^laLD<2!HY7dHE%3{6$|g%ARR?~OcXoA*<2AEX*qD%`2^Zd(@pemHw~w-g-$Uq6yzW!^1O;JY^gbi3pIaHjs*)6r$b{omterkAdOm&`N2gb=+#*>Yh708P>nm<$NJ1C13<3d4n1A_!h(~1{gU=P0n5u6L}f)D@DLbP<OdP}yzBh*(Y-gimmIH=CkY6MO^D$Ah~uR2E6{S<l&2QCE;AQOP!`e2?$LRl7Ya-|0y>h|J0x|eLbHeiixxv>!!J!X3!SmlT4~9i1Zcr1xeqhQ*S68>icl7*>SpM*wCi1A%!l!LZcTRT^Llt$C2C2fFQ@V_5Gwm0)di72FF2_C5bJH7Bpu>yuQ0K}Q#UHY(^+H<bKCO`m|ZOsM%7-O2filrg9(mo;GaP-{ca_r;h?G%YF~t!%h(64W^?eBc_SiZ@H=TDmTak_p+Yg=C|fHTOOkVLg29%d$Foc%p=(V7yFV(UXxcEYy;QMDvK1;&C^FEeydm3JA6{jP^`ky+5eM|*wI8h{?~vfib-cATO|>+_mje^?@$&`Bc)U^^nu<N0*4$*+M2V5pwzm!VxsqnGoSPiKKzrrJc~S6SLcOAo<H2OlPN~J0B;Kee#1~#JxC$n&az60fEF2x9TRpsed>G}2x7+UR-cOEqZr>QTk^Ut$sNREBnjww+pfjY+`fTI1b)7$}J+~srSLBlB)>UO&<=jsL<(p>MS>fv&>_HL=l0bLA-6#$Au#u5aKs=xd-F|TTuzT-5?65@aqp=^0L?}*&yk0FP!WIvy_LghW#T_+cH_rrf891hI#WWK1MUE7Y^*3BZ_3~2Cufad^LX~DdL7%%~pUvK4#~wTsc;|z}B2wWI?q?Q?ip50JU9irynsA9K!j0ZbTHwAlIr~>KX2ID%SuW_74E?X4pFBfT`H;&kwkOX|KY^ze*RdF%pMDyU-&0n#6kfY=tdXVj!*CAI7rU88Za<U(5H}qa59mIgh~oZX7Ntt4<2X;dCMxu;Fkkzxe6WLjk)z|jezASr$NhNQUNCJPEI46#%BG`{giBK8$6W1|nU6f$TQqceqeiWW!-GI%$VBH#Z7_<oEeJBJ9khAPI9}j{6K#v0gY&}DB2QlJi5{x<mI|40%nmx-k<P3@9~HXwrN@zeoC~(Ca2dlTQ{CH|oS<E53^BJS>Qt*HyjrOmdMw#wi(wU786R~kM^Y2g?YVO)Lv@P@Kr;A2jT+Enk&8L=*^png$)UutZnoN9A{T(EujRQ<J)AkU!G-;~feFasv5Xos*D5-G7d-Z&8XDrBuZ#Bf@_L%bXMR!3l9RQ=*4u2Gt-pYtJV-MEfzF+>NMjAO2+kT1xH+2J(k`O=7ER95D8zRFgr2Z5o8z9dMEs=0)!H5kelvsC?RBTRj`9-ysc#d}UPZkFj0(*m<LXI5SjHJ935KYHHassCCMbCz<=_ntFzjHjbhHdM+qAqeL|CTLlv6c<Xee|nP&X8-Wh&e3=Ni`M7f0*sRr)I!nb*6kF5Nn>!lFSFi=byuOa~E<cS8VneJ!mA0^0MK8WJ>^nWXnvf4Da{(SPgmb}MlmT$mCwd$OD~)#{k8%F?teG%{1!0jmcJngP=;>&|*8K)Iud0gjlH<ZctM;qX|uU(df*7nlHv`EA-f68_Y8-QMMUb7$H42ryIdbOD+%Nu=U$Kpdt_xxemXva{J(q_#s(pcdDbYKU>+3V7rvWxR_q(glpL5{<`%n!kakba^|@teFyZ3>*EPap?agVRZ;rCoK7h0_5~x0V>gaQ(s=U`+7<qaWJ&@Wox@1ivsg`x+95V(9$ySR)=Z(4pkctDV3FZf6csG1PsTX@N^YE8sV<i+qjh-(!3l7c1yX1vLDN&w|O%=+@eTyCqj-VAoFg`LMWF}ezc475vsRAjA<l8b|VOatqen3?5wx3305nHz5|Cahn4Cml(>rWX%D|ngj@8d3!QjVh%SCiDB~pWZHyGd|E)34v<WkW^ypv$?{#xrH$@N6-2=t5trTb3C`u3XQ;#Rpg9%#PEwiPY%`MG5I@yk(B2~Q`zr6M8tZy3ACC-^_Y(JXF^OJv<%^nT)#)`8qB`y-@r@t#}c<HZ!zMpMBJU{uaWXafkOyt>LF$xrbpPQQ=yL*26c>ytJ&%l57rD^7h^V9FB1xM3&_Eo}mNjU1M!vam6P0mlgHsmIp9{b@BXJ1Vb`5(XM)$$(fEO0d70So6Re~F3x8xBi;{Qms(3(0=?^z2zOVPkpr%<HdZ+s^RP0Itsa23}yCZ>J~rhZ(Q>eWs$Ie61R!QLAqgW4mw9jqQ38#QyV}Ke8jRyB83}Ocy8+mH}QgR2GV8E!*y6;KB4bW8hE<InZE!b3$djx=jvD#^w3xXBd9WJ=w8tLR<Y=%<P1ujbObBHA)yo&E{QOHV4IDi+J9a8#9Rw##XwUM%>i_!tllDLFLaaSyXy?z;apk8um<otxH=Cgs9tWh$VDbJZ>Au1FbI{w)MS;M6HIRIMhINgS|B`fYkoBXp&Kzj^TZ)i>dJ1`;Xsddz16iKW9}eFbHfmud}tIMwdJDS{JI$=`tE^?pKxZ!>5$HUkA<ld+*IVU5q9dx*yVi-#UE^;=h9z>5m^HN($ZQ8PmXFaDyG$Oxt;0dpU%?SJF~e>w=~0osIg!rpUC{E0SH*CoY5F#NRicJbb(1r&WqB_M*et68RNQ=^m?gd0o!!*;2^bvb8~m)i7be13D<K7lL#5>{$S~Ir}zi#v?iVa>@>ADFNr-5|$~zal&c9K!F6qGYp4GbHZYcXgO+$Mdzn~qT8>%9V2PJ#V6XK35D+Do2iXp0v5@_5{q!&Kk%XZ;S4x{)XhETvZ2R>vtO`d_8uu}?);djk6E|dW%_RqZ-!N>S69gqD2~wN%|LUc8+?2-$2Z<~%+0Uy3y--TdHPg$>uzl>Z-)v*1IdS%df*TBE<e$pd~k1zeYjAz)jIwCeNY43etj4K<xdFKlL;@kK!$z(Hfdr#(WV#V8pXzO!v)m1^ZvaDPyEOCAHDndi5G{fP$LenXkRd1e}Vgh{+-wU%8Z`X_Y(|-d|!9lzE8h7`+jxv$MKD+Ob8o20+M7T#}MSM0xkV}_4RB21)<}oc>',
 'crawler_snapshot.py': 'c-qZ<ZIc|wapreM@IN%cVDt<)J3G4oc;u2p2?Qw#K@#MFpza)t)wp-2XSa!&>0x)zVpog(AS>jPbmE0{mK{=r9VW$dSRq-W?9=H^gzxkL4E_&92qg6{?#ZmI>Z<PQ*;$CXGi@z;x~sC@GPClnvd*15_toTuBz^Vj^<bDqw<l}qY;qLLlSx{1gKKH<_9D)Xf=RR+Px4MMO-KB1kwwWYne7E>79_J#e2C}nxpU_pePo=a(_k<dFN#GL4+cRpou^q5M6+31L`9O$@<$(e^b!3w+nYyO9!s@hI+?^nE~M*SNf)ys&N{&;9!HBwF-nGoDLor5vMinzUDcYYbYnk@qtWYWI(g|Z9xjR$^`m?+OsDfnT*RXiKoK7nlVn#{lsD687VX8E6dOfFTqM(27t+5m3;6r>bf&u>PoregT@=a0^n^!86$V8*&=t+__r9LY&5!*k-$z%>Z}kt}bf<9<p>9-unn$A%#+#crZ{_LC{Fp|?zI17hM{9hq^B4J%MYB;lEq|8%%J*py<<E3@D=x~Pw2pa^49n*8qw=}n$p=1^Z|pHIX2}ppq{hR2Tx1g(B~O*^DBiu5CbJGb<nu{VxMj0+H!ZkArfXcMS6@md@$3BBLbCZ$lx68{U;w70`nF3*u_$8xI;?#~8z6v!zrJlv6<+3rHRdcDCzA+Bs0UR{M!IKvaWNQAM%1IPzI6SKtFK)P`ayVbDSY&iH!stpCcx`)9!J@5{|eDzoUQk|8z^)AwHIEyF}VEYm#*DdQaB{=k3Iq_2^Ph$)!usakpTZ?G5Rz!DBGQ-w_6%(fvLlGH_6j+ngL~6?GnT&*~9o+v`7_5f~DPjKia(Xlq%8Pj}N7W1p^Zj52$#HkWhoUofP|whU#rMoyW6QnC*sbn&YvQ@-)ccq?K_e2vrBdhQ^~JohHM<?okov+)^j72SaZ#nPor9*R<rsQl_^?N!C(7^8Sr%0eT0U%ZovJi+@U;BA(6ximTG#2D4}yGte$Q1y2U!uzPHXqD$D2i~zLNKIs<Id1%4u-p&%R3e{`wuk|g9=aXm{>;CGIswzbPGR}1OEj>)nEKtGjXfd7VEfmC%W;yZcC?6(C|4Y#%$1sVZ&x-zLyBp7lR$Jks7_U7YwykC-X*4QZtsn_ELM8H+wbV2N17fc&28+q)djM12iF6hem@gSi7<P$Cw5%~@z7e&t2;t3H7y!`&+OHJgC8OI<fllBxm2tH+H{x$yD#dFIupuyUlHQIp8w+PdSd--Iaf^nnfzdNC#$trsvS4`aOIN~<DkEU^!iP6A45GtBao*~84)`qY=8N4{7H;1xf}4w8FY4_)NhJdT-*$#(ir<_aU+A1Dkfu2`&xhT-$dY-h%@cV@6G?!&YD%~Z?Y1sc&cYmB1sS!l*_+i&7lo!jdfmlu=}$5aTJ&{22(b+(V-R+i78pZj^2-fx<K9OJHjS@FlSTXz7<SeQUt1L5=Rq_fbUPyU5k!H8^eo1?pk0P1cetOX_z?xO_;#?ihE}_J67zJC4P&Q^>XqgcW1{)DyRWPLf%cu2Qvk!5kGuJBA40)E?InaNdGJ_2*bb8sO)QWo+@X#@Gy((Z7Dw|q2SXktkU}7Tw5kxsNjw>~sCJtRFQ18*4(AZ2V(dc-T#$?)hK&MRAV_X#odmAMbe5Z8C!<`AOVMhyYlhX)W&7OwowoYN3>G!Jv&CcrAqxT<4Qf8@@-vt=goW9#XrWBI3ps3l1QfzNOu+i--O^FKn&i0%nxh2}ZHTQ*FVEE%%dT@<)O(Q_Tk?LKWZ>CNUmp+Ny#7iMEsFgVeDo+t@*rP~$H_1O2V^v#XDPNG^x)ax03=E~X)wk>kAmG8-|-_ZhWo_Xr9ITyjqr-d0}BZ5oDKK|_*OZGy!Q6UsU9r}YH{hVD%<K|sZ&hZB%vM7svr|jxRYRpAeGI)4gjJ<E8Nz21;U;ZJdpO+_KRYke_?BVefw+c%nU3RXa84Qg>iUw#-rCWk6GIaj#2%w>_o4C{trSWlmp^$*+lb(bu@ONy}h|pGD*9NTApoHb0>JRA8gbHnlGj;dojBQ)IYF;cJN%V(d+d*1F_3{paZHRZE)V;d2XQw8`D^pK|@QC)yz^vnnpl%41zu$y!7@Wnyk^T0roUWhpY}Mi71PSO0r~k!S4tes9dhTj=`U3%MuiOdjVAn^Gmu=m!!>G@lg&s%|PrjfOaS7NxQfA6Y2`a#`T<OGb5`YqAVf)pX=1Ij$fM@eXhEI%w|?@QUB)xEte?+`S(R#>0al5geAxE$ZTT}`VUg9H%&~bE4GMkG*ge`nTJ6vBiMFrVuq03I+{|UzqOW7YsV$!4d-K(@E(F}Xh2JdW)20IB>_D(x~yAll1vlYjh*SQtvE4j0!WL<H9QMQ-@DNqBK@d!O#8!@nJ{K}TLH7DP_@Bd>EWa@ELa$P=)rad8V`6aAp`!mhwaIlgSMyASn4(RTK9aL3+|wWv-Dk1OA&lO5GrhCYV)8KAbHa)fgy%c80A^*1R&!6B%1Dyq9EG}GTCMYd=X^@@%5B+4X6-NQDjU5=ut8n@OR+KHK=s}<tvcR_4iWF;%c<#@q5~mTik?xvy|7p0qG(cV;fAtq1M)t(V7BBv@bsasudp&CyP8eh})%<F%C@2v_(-{R#SOAQF2R@B!c}IG>TocC4$uR{jj4zV(|LqbD^5B_Krh#Ry)--J3fCtvsak4fK2bzC*_Li9-gvl7Uih6c#4}ipl{QCk!80GIKzVMu?6accLdT&Mao-S<1Omgz(48QJG0(2j@9tCf?kEwD;9oIsnK1`N#tp1ril^C{Onlsp}N#Ivi-{1v(olN-dA8)+Mm$kDX6lGm1?do&Dbzd@|miIO2=jYLz*BgBY3*Dpg}~P4%EC6RZMxAA*kZPkxu1vbSR&rL-(`Qh!IM`DyX#{>hji&XId_^AM;z0Met2M*pQU4LXynDEow_K9X(BzGQ3y<?WA$eT#d?|tsy84cr=I#;&o(0B3i(t<uSbpFJJxgB3s;m+4pc=8=;8KU^5*jdsJeWX7fc(?|A#7S8OK^?NO>_!#9XKLeZe{Je?fG*b_TuUB$k$W!aHL!E&I{TtN7ywy<KFT0$?6;f*B-V-myE^clYFbWt-@fD_)u-DMW6nd!jrHU5^)spk+!@XLrM{7P{OK?XtT+6cVE^-+a2LCdh)#RJSmWk+Bwc#th-dQY%;5!CgABwgf=@@p!zZ37+N5trWHafKu@wi>Qb;d1GO`MPdno%#a20qNj*eC`-^6%Re(5ufwwI{@M{29(;w3<~EWuJBiG2aT7eE5K>j8c^j_Cx~C09+4`bKPuH-f8}4R=uzQyHa}c_ah=r8X_B*Q)(x9_LDoW8N2A*VvUrh=6r1a2uNOSmcS?|AM>tzB`SnOfyWmxb0Lq<yJs{J@PQB7@3PVbty%pV2Oa^oEJ+x3o?as<3*Y-<aVw)uxy#3h?3exX12xU#vM}r^=X`VnOPY=;)*r&2+sM_zC#@WvU(*zk&!kzN{IGKSpsQunEd1}UGM+E#+%Mho5S%J%X3Xdnz-hfm;)PkhJRXxSPd9cbDjN%BZESaiuR$I2CTd)rZs>}R{SQ4#vN5GysY{eM{q?-1L&1-yf7(J|lSvmwF@t)J?9mfN?zKK~3gBSss)$|H_dq#cHT8T~ER$*Tw#PDkSrD48{nQUR+v4t`NN+eJ}G&i2Z`c~LB>*J<fqDhcI7o~ABo1Wm+7+cZ4UP=1D;UvoQfPF)XbY40f#&fbQXog9a*#UOAq!<jKL`}vlL4$#itF?e5vk%*|E0x0<$t6f+<_GKRVovI5*FaP-h-IpxPQSiHCbKwe?g_?4-Izp2{f#apD(qcwfdF?ebb|Bb7l6`{56oNI!a5!vBIQ8sGtBRR9AALk-?&6D<)hLs<jaK~zBB`Ztvov#>@G$ig8nl-3(6RSC9gwKOc+RQ=6FQ>#KV$jJp#Sg(t66k%RyHlRR)i$=8^V|%2!e4l;~6W=x8a#3R}Jg+FFE4lwH@`3D+>@XWrF1(*+D+yJUZ#r+HF9l+u=Jho9T+DOr;>^a?a02v5g4{+*@#FZAOGQLXpG$^dP=DxukL&j6}KI~!E_c4bf&o|cfTmnW_y2ko{Ku*E{u-pWdlv22?A%+i)gn(M<9#>OMDO4mQPu=d2r(x_;gVid^$hsc<igOucNs!I@pdaGHlk1>!FgK!EQ%TgNi8Zl1R3MkiVKnUN`HUM*$%=uz10a>_w#p@+2jqEKVmTya6^D7A4XjS_H!}G0NTAx8Y0<E}xu?whIL~th^>wo^?!4#2XM_jnxW;AiS#&49DQ11cM?f_UPdq5WNkxBfBZ7ep1u}Qofah30Af2OJpMXV5WaEr0IIy+7ma)=|?Be@55lcIM(=wvOb922W}Tqmp&L0PCPMc5XrB1gJ!NK5V0Dj~ih8o}D7Wvk=xP3Wv^m-iscfrQ_@j4oUr(&o<kHWqY!znD%s5O~-w0TAo+(fHZ7`@QZno%8GG`RP)4(yoPC)En1x^F@yN8A8$APe<i+X@^*=;zv?ZPs)^C`pcc^R7I3~GnVZoP>Kv05ZF@{woB(Im$n`5@(PkwR(Umth;ciuBL=etOC$;t<*i|P-`hJ5-_d<zAtQJid;Z0X7ush6jsc-I@w*X`D1o&bd_1^}#X1U#RJ+%RHN^)Mc&D`k#NyEl<!uajk=7we9*{c{g6$ms`9k}LH1{Uv%zT^Uw@6c@u(4z|PFs};?KmSw)fUEBU#5OvuZ)r>p6<1qJd!#>t((jT^4OyOY+u~j?(I<6(O_?q?naYF25wL0VwOkaSOW?>RfKSeSZHjJ((a;o37hWO{;$PKwVLNu3?GSp>ct4e276N6uUbWzHXMAnzR_;d*KuCT1xsvtkVA2r6_X<+nFr$3tJEg8M-sK>&4vXUP7Uxbdg@+k>8>(|RIeKi-(CQ7Yz7lgWObYZdIHGE*(+xtSmUYbPE*uK5vu}Lc{<i(dFH@$L~02b!u{)NK?1dtLG#>?uoHtp+>WsM!ntP|qN!^(ZGeSW8l_5@FX^tmjj=6+W{XWAp8nTR)t*RWB_!hGMxO64n~jwa_iXdAd!(mf2w`cI-kwzr8{CO}636ovb+P)jHd)4bM4NAA06tw=QBEoJ=qF+cPj_H?YB2J6=CR4BLs7CEI_%6YchX;jtzo)SgUt6it$@#t)G#qH`^PQv8;JZ`c5yw}RNg%!I9ICstQ<zB(IQBjv6*>^rqBu)j`N74sTg!9q~Nx(CgnQx4Q?Yn1r75y++WOy&lK<=OxT<Y`tcVXbJP@yi&K-qDv!CC)LKtnx^&^G_L3!1%{-~Xe;8Q1xiBfFT>kl`?%U?&PqAr|GmKn<<M34q*eUj-8C+=u?j$JfnM{8$WQM17r$R*<chSR(mzYi(nM$py#bDZJ9$6i+mvmgMv9^LOoAqT}aoGU=bVM15z-|z(X`tGEB@%N-8?FtassaNFyX=&rIH+hb<nsgFD{yvQMah}c3SUgvEGxAIEJbzb$Wt|~4c*Z^TsAA)_{0AYvh<o)L2PX#dS2XorW0K1U08-BirH9$)p`Rv2(FVVxpsLBN~NjpEyET6F0y1~`lqPxqNguuA!cKwrJ|qG-swGB+AHrN4C!T;o-x(adCQ0l{CQdbtueI9L3Dx}u%XH0Yv4PWjobXQy+ZFvYiU4hOT^hG@n*-fnIptm54dRyQ#ziHM%)_g1etgx313ePZu_LLj<2IjsKJ_ae>Yfk&0sGFrB=-cu)iovgtj7Q$zC#ZqFCrNsbu6#;ja`LzOHI@i^C$cq<Z4&ti?79PKHzVP4OoiEk4%JNQr}UfMfX7iH7i4_2*>k*x=w`2GiO}sN>Yz?d`3Nrz~d=91jT8=P>2tP*sNwKs}hKB{O8a;dZMoe%%@JJ<DT*IWg2M6Gcf#2*t{(q7WqZfk#Z?I{3{Zh-C|!r!Yxi!DAu{^jhg$+BOMRIwO(Su}dgeD9f;1{Z^W1S%hQ3HadRm^pnAcM^8QDZl-5j8)A46{qVZ7ka1V*@}!J)a0{_z{m{mS!BU~q9yy739;YmJ#aorQIqBhcWnp$oA3@Dl)-=D)i0A3N6_x{W1Ia8ahRgz!0>bX;ELbe~2oX+fiST@P$OBE07qqLpb`D4?F(|we9;}Y}Y0D^K^D2suilUvD_|Yhrq7MxIScpa%EK!`oI@A-RxauonNah?SHs^wxrcRpL(5qv}Xh#cV+<wD`vm5VmSm^2M+v!_nlZ2$G#<rW78QAD;UJTCjKb_Neu~Yxkx{SxhZdqT9RNUO648gXko+8L;`E7|@Ld&4(+lr~^>v^-|2KtVGbwzzyBN2cktgyZ6Y#CWt;PsDhZ3PF){Q~C;VXtmUye-7MZs~|%e1GU=Y3V?SwL-eBqtZjo$=4z+GZMYxseQ8QzzDY_#G`LVfwY%t3b(Z@RCa}Ryn-EgU2l|%o0MB3MGb|=dgD47M8`s>Cxh%5lzGzK`+C^ipmVBBJ8CJ$f}$Nb;KhWys0flNA#1<aJ`IcP@C=p?GLDy_07AH&4jaWw5i5<mp7eLT+gcH6)uu$Pp%UbUOTFGsM}Pv_fc@`%_VrQ$`0L4hIvBK1(cT)&0gaMkWiBA#%_$xthqgUKOvDbe@&!1bN1bMdM{)1V`KTUpg+xsVPv5XGhiauql?b5KP};_5fn&YmKq6W6ehmp$QuHqLAi`YeZPK3$^ylBvpQlyam&0#f(S*88w<z~bIZs`(bpf_r0@_(-KJZ4=EfGbQgz-061Y8-A%mG@k62U+nQy^51M}DCRb1AWLGNnyXaMm$NT0ahID-VJ)>*K?fVe_Je&yklc)Mp?enImr39Mj;Vo6kLlurJ21hEhxEMhgp;*hN%h{7T&XnQSh#8UD(RS6>Op4-Z@jQXM<OYEZlBm;?#%=tkwVeapyYODn1~{;l%RDDCliGcPogKs9*y^x(s^ioK+F5nj5DUVH0LIgD)|#nih3ioV!vx@lbKu?p%uR<KUOGZGSjCSvpyx%(?OZoD2CaP70vtEDJ7gM&{1#wo`4OBK?4s@MB}JLD3x&oE|Xt+B_alv0m6)SwdiwV|`?PfbeQ$^wDbs_|6^>bZe9_|UN?t}XHKjq0~QoQI*bjL<DsThl+7zTQya+bbYbs}c;vzzu98w;*2PjaL-Z6?SUcEL)90s~v8yDn1iWw*#$pWPnex+g{t)Ii0Ol@V_pWHSm{ZYs#SmB?geozHV)AVn%gps1ujp9I<@UYA}2>lZ=2cvU4fTdczIM9$RLie&1^i+FFV1Dn5N9H!bt)OQFiL??%I0>6r2bYzF7g2QA&bH5jE}nUZ6z$Ob`aSjeT_-MrLpba@N3Ki2X6aQ<x@(`i9rcEvKqvdCA&{=BY7tYo#^a*bwnV_A4c)0RtPx+gRqk9I+?>zYKf6TPZi7`*D6CumcyGShEVWW*#)9ay4Z3?rDl2Kz}N9@|=2ma&>j=ftMyguJlaq6(HFFHT5Sb2OG1!fF3Ko=d3D36cch$1E;j5wLe+9kQ{^9#uNC-FOx^zOZhc*Y^E^;Jar^rTL0^kY;dHn)KaJRln!*IZxywlaV%`tI*Tgxb0;)s>vo7iGsSVK<$*lJ*9&}2JbsU?Op7k?}O~!s#SRvLFSMv8|9nM+En<Ro`nW>H#d=24ayaVu^K;5J#_Vt;$O|Nr`D8v{7z2Dte?B+E-Io#1)s@+x_{wp0<;I&NGR0P+#n8G$5JfYlx)d#{-xDq6@ynxvgVYi)UIX+cxX(s98a8Mp#)JfGesetlAtU6(!8g+Y>t9eoXu)bKRN+%vB!yA(-TCxF+85_C|ZP%mB?BP`4x?C7|jN3WNcaK*u42unl-W{=MWv%%3Kbt9H4#E7*!tdOS?Iyyq<j|pGj6KY3n_`b!s_7;yT0|=t0-1ibT3QM8D#q<>In{Y#z00*LN(bR4!GyQ>jU`qB7}to)i7z7_F?=`zkv|^|0xnbniS}T?@Z6L=Q=awwLIxcTm3|Cz@Efv`Zu;D@%hDEbgK+ohUo;Z34CUX@8G2_B$9Yj^H-QzqF&<5xi<!S8w#*-ee!6b=lUuX2(l;OH9P|>gur)YOu_UyUcxicBkD^|D5ja%8t2aH`_zG*u=r6{cH+F+e>_v+m0r><2jL6jZ;}cVPEcR(58HkUUs3Tms!|ZRmYa-Va+@g$cm8uTLaCmU7Zl&mYjd}aM*y$rNGZRm~^2rZmY@IT{Z0~>lcM)_2|zg2Wioti%JjILw|0ikv|^1l0vdpel`=$1Njv(i(1p7GXfE^@xmv^M_!fhd@<{;B-Q;9TC6<<XOP?w75UT|<8>)ci*5Z-v5bPr;ejTE&1jaKrdTPnqD{8{v>4z~O__Hra~gX*Qe~7bZ)=~QWBHGFCVQO1*vR(SIUME18Ztm9V`W+{y~XX5x|;O)yS2<Y_=ogn@m!brsfcGd+eVS%r)q@!-HNwv$v}MjbSbK+TC1m`U@^%Jz4JxMN}LIkLulvcO{bDcb%I@eW>SkH1NY0oaea)4eOxrhyL`xGmjv%9OJbE)j<g9xLc>Op3@7m#^0iGPaFhLbk#TxwmCswS3g_o6;fvw%vA~dzYmJfv#)pK@Zf)@2P5yg<|6b(3m-z2f^tX_Zf}svJV2G5&CJs+*dGP?Y&H`5Aks~8M9!w6Wl)iFGqt9_L?|{w`L&lo06%YvfZ15W8ag1pRMhP_&Srv$<l6WtJS9wfp!oOu_r5<zvRwY;To;kD*twejWKk91jdIC3!KtnmExO|A5sRd6-aWr9~OU4=0YkHcc*5?DJL^2hIr}d(2K$Z(kfO`g8-EJ1vYSS8P>ag<3n)#rSV49ZJGo{+Dl#UFUln{HyU@8#Q5_5?<7va}k$|c}CZ{xIG7Hyd2f|(ZOQCU9r;7$;>o9A(riBm<?$TW>Q`F5?0zY<HX;MiG+VMpDE<#aG#?CPVO9KPJbv5(jy`RnArrBFM~j|8j3TqT<j_Siw5TaqjYM6M(z1kzH@CrV&MY@wuhfj}fqjdc&_9Fw!Yet!Fl$*6x$ZxIM)DbDR!iQVW>hpx=|UN-6}y{~b~d^d<o*TLaH-B{POR=`snukl!Fjp?YaQ`-`Ue;CMo>pcSv(g7A`r`iCZriha$WA`3`tss5swY_`q9t9u$;G=KO_HO3qxA4E#&C&5jXY*v;`uE~VyY+>~ZjRb+o@MDSLtde?j($<i;ttL-=OA^4woM^E;S;0vC#LI9{25XMGmca;yPp<z@AROGfU-CP+ORxZ&h>Z-K{Vi$0NIoeY%p3;20&~mw-b`ga!EvLMSdbrH`{{fnHjJ_Ke?V^eq`=kz^+EW9qi;2Wa@o|$i{qbc*KXVIPIU*;FTW5`4FTU%_x18=al!VT))1zb97a#Yzjq7)n>7VKd9r(l+j!1^lis*q6iWANhGzV(x(gtNCgwGQFggK%;cI-9mkVMTOJ-QEM7AK763wC3cF&}?#>r{pnZm6C{Pi}`u@djY8&Dc<uIP1L#=bcxlYN&*77+eBmW`;hs_aR`7eXT^G;PThtug+V{_3%w#IydTHF32PWg{(jh3@a$tYUf2hr4KM=iq{xYIUM&^gT^<#U=Fp5Gf+uLZAMz4zT~!M}g=^0n|RL70S|YGw#OdFCr-I@Qt28u46>G9;fPo{Vm2Ip`LsAm<%`lkT)^sRoUa)sblo1r#y1yUu1L<x9eMvu4@KiDUCbPdR={1$>iC+1d@Cc>TpMtqHDIk`O5Tv!Hn*q09vwQ>awqic?A6s#_!E^rPw{<)~uiDWrDTl!1rJ1kQ1*3nAThx%<*9zLSBMnlgbZby%I?q_3Q!bu`}bitxuF;~eGT8(E3?nCR6PA^MKW&47Z|&d(Y470z+j+D*D$C<_AH2a-I<ED%hpW{zKQie838N*mPMqXGXU9R|sm`Ljw5rv}F(<MJnFJwp}NCZP?!wNeU&j(?Mav<TZ<n@$Xzl+#gg!D$t?mgz#s0083SM6ClScGE>qlWY6+j7$fN@`&!6L}WXG{s+q^E&(t%*gz!!r&U@AWs{?mhR`GM&m{Qr3!4`<HWdm0o30#X3GSX53grkaxoClJVu6StZy_1<@xSex=Wd2OUubRLyuG&bWIL=<5^7#8r2}>=s`^`w+4~{dQM0@8)P<)n?wIICz7c~}i#_0A9JZYv=sP*kl0fh7VRxQW)@oxN7@PiZ=uj!IkQG`3U|D-_mu8Pz+uOZG{~WH-gT2UBizu}5bnik>xF}%?5`xVaIzm(HgESC0jWRhuZUt#s9oSlyJ)j0!Pmiw6I#>p;mc4msBkG&#&W;zkik$R&B~5QF=H7FDwGNlDGe)Jht`6#Iwp(_R*=?U4B`IHQF;F61Ka`AgeC3ShJL)9(3Of-eHXRY?Q2KCr^NV`V`IdF2Kt$W>9Jfx0f=aMx-Sr|GjM~6>c#I`F8;}|R8x1tmEO{Y-Ib5^tW0++j`cmVTE|;_>mN_LUQMoz&W<Zg!%xP8CB@p%}c~fgfGn9eG8C0^+$6a*-=)5R-tw{!=RB|P3clXkR?!nAf&siGgjb2|B2K=y*JSI_?Uh-`K>V6LkiR@SpUW09SHprO2T-PiCfMgQ2heCRxT|@~ntR#}e)M8r@<AAXv!i<GAVO^0W00`^L#A0iwg~FGmXKf9%3iar$Eh+U(4UZS3n$WIPI=pI;s!6$R+-j24-&pLVQq|Hy10_aii3FCBuWb!p%}F(+Nh!A~m-kqw>jOwsqnEab#vdg3Qx$qr+U2vk)$>FOWwe2t8tPZ>HR=S0@cJ5!2h{t6&2T9<l4P2EhJrViKCJ<D0Wjqw_t5&PdcPE`^oCGaW!6k7Sv;Xiw~8dncG>k|>dq=%Po^Y4^|!STQsY~m4HZ+F6xLh%C3)o4TH0zUQbtYp!uF1nu%{MKq6t{#i5aLn@DN!unNZXnld`?Zu#l?iSU0oz(#~$VY{^M<%=QZBwVf!JlU^84wXZ~%?^|QJfM$GM5*gR(*riQMFsb2)hf-eB>keo@meaO0oKtSr_;y%TuH4$C6P2`PPNhbvSbf=}eU&J>m0Kw%*iQRvUHa7#1(mqLP#<dO1eL9ieLP~upmAAdcVZ+&1D087_!cLRf_NX^TwJ)=>xC4AKcqi+@yvKUR@a^l^%)dnhU612Q*mFA^@Y_40)t8;K66I_An@GiI5lWkg|gQ1O*vf(XqoRva+gKff~O{(^Pqv`&J|+0dxe;s1~GRg=(2n=dsDlZ(Q_oKCukdRIC8<5u6db}{!}(iAM{HC$UR`5*V8(}8*Ek&_9=B*K-Zo=3w<HUb=nj5CnUQWn+zrhO<0gBz&i<fz_PBP7+M5t%y4>kH!b`zb+IO2-ycwWBIt57cPZG1+aG=N-cLUI!M&f|`}N1~ef+DB{^H(WeDwW$-@W(4kKg_HgB@*9Yyf?bxS$>0VHbo@1lEB7Z8zkf_6=x<t&Pnd&zPxUbz99K{Qb_wIE1E#_;SchN}Uc}<%ZF_vD#GK^ED9p&(yRL74zEbZ8U-qIV)e}bR~ubqZIaB(TrJ_&`!_L&b|;&@1h}|eJ^fPf0}!vB|@5L<-n=1BOoBQGPZpp-!lbf!T)xP2rS*@AOLolkOrGi(_69R)9<L8BPe^C`0x-c5_3$h<@FQX)BthNBK+hx-+A!!?>+eK?;iZ%yAOW({_p?wU1Bv4zV(~m-~F3UzWdjoeej)6zyFt?{_vmq?f33}`oq6}@V7tx^uymh_|`k0{ra~L{_9Ua``{<^?UNt<mrs85_n*D<!Gpj02L1l@fB)dqpa1y52mkunw}0l@?!({z_E!(y`I`qn{^^5X{)l1!w|5?V`!}Ed%a1<$^{*fN<+mRE^0)N%qwkYhJ(%s^|L_-c7}@XN{bsOx@1JJKb-2JQ_7Ps$zxNBg$nU*}1=+p-;rnSY#W#DUX))EH^@{NVmak++n5A53T)W*A=A?`l#e+_o4?Y)kFJ5YdLD{%eNs7r!QWvM0g;n^3TVEA|gwCwXht0N$hvF|9LBUuwm{GvQART5EEOf}jFH|%GZmQtvt5MA)$iR^k*<)Oh9F>e<0MukfebK?jrQRvL$d=P9vOpEQlILm;LKi9*Vh%T;s49pW4HIjdY+g-<5Zlvnah2{UK|B+#XPm$cliiu-=^DFz`KeHk&pmc?8;vF7JJ0cp=kGiR1@L*<0>pCDIPR2P-^6(c-xqTrVq}&@9nP6k2u#K~X~VCzzOaQelXq}V@eT<<NYSp&58io;QYPd6%#`@=9jY>8TbYb|bmtH6{f|HV-A{i1zutN9iywhs$nQ`7_Gh1b=l?wT_OI>~6w?>n+Q0W}Vs!WK{uQY5{)g`+cl4`!|AfnuLG}M|6mZ#g=$SsJ%OXZ;=|JX!rBK?x$fl@5T&>xM-;?gDI!YJp`rlt5v;Q4VHA0LNuW<()-JF8uh-Vx^@K$sXDYnhP5RP>V`rVyB{Oo(5eejJ>-~Gm&VKhf5Yvis-^OC3`;Qvtvx11x;F)T0lQc3ko;Ztv!N&u8&dRoe~w^V&itOq)g8Q0%(avDifS^So|a=_6Pj;r$RmWgMq2-{l<Hr@XNVl*b`S=Cmer3fWHY#FJdJY-FnB^+|g-t8%##Acscn(tPda2TZxiFmA-k;?mZBx0U7$?65bC#49lVz6P~#F=2Ef>4>0GPc5%mjcU$GIVbk6$<6RkDFhqnl#DGY)FVfVu)CTmM3~~O_>r@D&6X;sOs`A8qa+gl!<^ZP3b#VvbY88d$B{Ec|+HPxqgSv92iJ+$N<?jEErBY4U<V@pss6PF`Y)`DiL!{OkcNaWm(V<R$dMDm3z<^i_VcJuHU&try5P`%NtO4NpKxIAk_=ZZu|4Ql4jXeG%H;}^LTJw#~^Y{Ddq%-!~vu0`{4HM_8^%dVAd%nK@I}ufEZ#t(32*qx%hU43vKk6Y~@*I!*HTnlzlxa*&U!nTb5^D<uHfIg5pL$EICN+g$vJ<zG83S92V&#xW0a3_XOAC;>AfQxgOd!uh5|B0s=9<_JvKXe5lZ5M`^s|ZTW>-71m(V<<q+g`&;qGZhc9Pd$pGL3U9M{opQ?=*_(9L?aCEtJIimSRF^t3f5Og$jc{b{c94TOsX9_lj2kxsY*p??<4XeM#(;Kt35L9(P1U_(uzCxLo71j-t%$mOrs0;5%KbA97l5dnXZ%-s5QGyQ+bbK3>=NWKQ%^0<w#Cb+@0y`!b7@LI8=5^==6Y}v{|dh1A>_omDP<h|>Py$(xcb^P+S`<#>$%KsFyQ9_Un)p#>WlJJ2kNHQ(3;aq??OuYKZwd35InWjK^tAhIka{DyxumQyG(pg=0{MmXM=L6`4Mh#Rll?oL4{wDuE<fDs*L{W1qu~O-15isE%E8V&{T#GNDJhxH;(#NDM(OrM|>l^3Z-@ZToN46K9SXzJzeNb$u)9&3ln8|Up;N4`dU2}dP{PqZ%maM1(p&aGjGK4$aOtJhtCFdd0T{C<u)Bz2-Ge?5n|YYBrJ^L!w^e0qv$I`w-$?2l7#Mrq3hdhT(n$~oIFm&*>ECb%%7Lh$n~=9Wf6N#UyFl|QBsVLBC=UFS?j_>>WAaHhkC&mGnCW6>Sc%QY1h@8?mEn)gV?>$u&m>W$EVT_*BZ*VhrQMCN;y))B9>il$jy20HYCwZR!aJ0Wzbr7=i|~&QCc~%FFM`v_*Z0s&3$p{ZH;8k_2U;efgby?lp_heRB5|+%EL+so6Bwc2)@A`dD`BxOA0>eF#Os*<s1NYgKw{S9notq9mc~2sHa8M>1-68_F#DOL&ZR$q^*)JI6WevFW=RjFLtLCd##lnIu%*fc5Donj#rLg^1ay|0nQmwa1m+D2T>#9u0(Kj!LqcDEueOukKp$OLRSS3)UL)+6SK{@sU=9?wddx_?<a!IOJu$71-g^;i^HH~$$WR1*RSU@BN&i-03OK$ead7@k3FRGm7~SP+~KENFbbaz!=f?NuIv-P2KE7xLfD+vy#{$w#W_;GOe1qieE1hxUpOovC1Eegl+*^U%GniKmX?Z`+SlGtmy96Z_5jH+bJFP{Bk?=hl~sy_gz|E5US)(p0kftN2T0*w8cqQj$Zd@f{WDy9z~e6ic5TIxkp{d@onz)45TS*_yv=X`Ul={Pt3vg#L5v*V0J$Og9S##9_?jK^RgFvv@a}-P!&**r`OXt<Ba^(x4ZXKoJy$cdw{G@(Aa3%)X4MxcQ5e(wU|7Ag!hbGQmE@V=$PDQioxe*-_?Pdmygaz}!Yfx^Ci?ImJ(h!-&F3@~7rYL>0g+Pp;AK(mhH6s|_)H=DxqvRuLMLUWD|fdCO8?;aohj|Wo|Baxt|>Dz9?pu@GV@YFtPJ6*nvIUUd*!t&FArY&k2r#1{l50XzvcO;5Ai68C!^CcnKz-~nNaa;*!AW$KX&=^R-Cd(Z(g&Gs!$<{%&e)6O!9^=87^jvNoAc9Udasq6bh)^8Sg9ayH#G>X_ac}3Yv#%e?0vBM9lwx+VSnYBG;M=#Hu%W&zmG#zT&G<q@6cOaGA3dbI*0yrQljN!54B{QmguMmdm)=fV;|V+*M(9YuR1b)r+#VCDavWovX9e)z}YrEB5%@l{Cv1bCz?VQ_5VOoU4yR#hkFNqP%OW!<i07saeFOTeb;(wYj}P2Eyf+bQ8iEIa|JI4+tS*rX)|TSy}5)aZ<G1$ekT+?-Rpf6D->f=@=pf0Ea*4e8`AxKb-1@P7_&&j2NP+=?XBJ9jjWIGzELRI<+Q*DRZu<#%LpTt_Xo<t~+C&>?!F`Zko<HK)!_^5@yEAGKbly7VT5t98EVOS9q|ov%-+jCld%f&~0)e2-T{zK!dK1s{^*8;&7ZvQJyW$ahSVljtBS)4ow!O*D1Ty_7n#p&v6I0#sO1Fzwp%o7D1^)<V0+Wp9SY0b8ybrfSIjw5VDd|diQ8RQFRrrtDNe%BO#qHCd%y2mpJky%f__LSA#3J8%4g!vN}O>c{AdxCKTeUXG^6cACzfEb{D6$?OGKXNO`ti8rU*}3Cq#YVHxv-Pl21;&*`VpwnUSn8<(0RF)gxHr(BxJ+M0>wAj=p)1#uOovodf?8)Sx}wS2qZHY}4`;dY$t?JIUhy*k{sr*sFYWALWLN8lfRo2lI?r4)^HzyzlR>1<}%30E{9BzJPLpO=J=u3Apt9S_%NS8W+f?zpe5H=3sR+B@o$mxw=ArXFj=p0~}zLs>cP<Wmc1yX7Qy1a3Z;K72rJ45P-&QFa_$;iJ!*-+17N1OIfSl7grA%sBB^fhk2#hiYy~V`^UsO$m|@R5fP60X43$Ci*qzU+y?mWwnL)8gW7l>K#_V+&BdCAxP|F+lRh>G{Cn(xdW>0POrQ0bx%Ew5}cWBP$2_@FCadfEb3r1`41d;0Au)qZ0Woh+R&xRhR?Xdkg}X7w98{%o*wXy&Y@R0a5PGynM8;1NjjJ@`CUu6g_bawbF2t;{>&vcEmUcn6L1dWWYX$Ba|!H?19}BG@qvw<RhxjW1XLOr`qQlC+0xD>f&#f6$Ph+Oh$QEnAsm^erUzB!>7>c`s&Q87uU$E$C4A7teO`~zz-Tc{D9}iCD}X;32mb=PY7v;mY8uWObJTq>hY2eXKVSZ=q;a6!Puu|&%_QfD&B0y<YP2+G3Pc)lB8lXS>;7rLCdbE-;*y<8TE2{p+%<!_Yc!UNZcmeJYKQItF45^39!9#+-W3DKr_NCK;RL|w5=zu&%6gLu(F_>L2kDj+$GIiWW97R`IQ59!1#Yy!G8)K=uhMM$r0u`WTQyiUz~Q5gE`Th|haV_}L$0;fdo1h>`JhMFAa3d}u=vi{0O5XUmSBj^VHY02#=jdA)r#kCBoGxQ#hLsBav>5Yw3R<4B&%{A8Bg_~K=^p@^8F8gSKurK0)3T{NYZ_MHF+UPU%h&rPfEZfb#5)VvVZ^XJ2U9M7M7mSO=Hi3fbPA^F^B(9$ahBRlV^kBy?>$Eynpxa@IiYQjMxDK#<}V&P*Z%R07MMCjuAU{0Jg~P{Q~jHhYPF8a%Z6e_wW8QM>zj@M!r!Lo2$5IJpi*QG<4z0{SW_f790W{CKO6(_e6Lc^<#zxl3=tzBfIzS{%6=%>V-unXd3H{`*;5*+PM3l+-`!Qkii7!6yE3g<C!;58}_@Mt?uR%4eeapzxUs#G;zO`Odxt9WLL##4(@;Wb11RN{ktEi#SDgo7gt{l_U_;P!LSZDWU%kgN|VCm-n(UTD=h#1$G?i`GT<YQKmA>B3&6ig-JrMk@BSkY1QDv;;AIu5I!B@R6O39FWB&_Op3pa-*8%(E_qfu%_vYm!RjhUj&ln?Qo)ORf^MwwV1&(0FAkN?m9eM=b3=tiOz7oh3ZgWCyQGL%bHB=|zPVgV%UCsh{hgS2i0hN~SPVg!>bSL;qNgh<h#$Eirtmp0y5!;<$ZEY>!zgx@y2`zQj7BK7sDMaa-ZH~{~362j-&V3T7-?oOM-<24p-;^4Jd1V=M^s7=g!kcq8zoB#2)QfFCYxel#m<CKdC(O9_ULtHJ{PREmbJ@-<=b*uQS7_epuw!sm^f%k8YUo=YwMJ<03Yd_hG2C<X@vrXReSfyMTDz4aaQ=3U9jP9k86tVfD_3v)+2Hz1ufKNvMt8~^l{tf|PEgg%f=%KNMcR)^Om@xgNOKz^5{F&FmFQTpx9jr0&>_Di-y`R-FH@pws@2_4n!0tV(SqbN6ZtGVUeOOhIL75rbt~5rGs>Dei5c0-q{Yt@7{QPR1F>G5%0W4gE1qP_fuj7ijt>R?+ACSiIiNWVM*@wMQ%i-D@XdD_r61y)1sn=crZrY+<tgc?95%Z8V(A1{uR1kPjjS=eK<KT{frF9ahtE0SGumrMv_5+wW0CX%CaTETo4xKK>HbO~ie=f)%@3m>=1bf$G$GSl7!6{cGS1Op#R_nKE5rv#_CUK=odzG4eF|WIQ6!Tt<;|i3<@g9>M^pxSu1ug7Fja?WJvqi!Ri~(pbaFu1=Q}0#IWey0?ZAl>?0cdF8#zpsvFP9Bx>?I5*{8jML)Ij4Hl<pvPT;27H@0}r;Z8O!G1709PAH>|v@924ZXI@Q#HE$JgeIty(-Wy<)uKV!V-T-gySq0@cU$3k!GiiJYS;5dVSbJiLx!uY`Vzj)o(c90+2^*s@$w#b8)h{JHxcCTdJkux;%-ydgqqRnZj|1&$_+f~ute1@NB40$PPm$qXdjfY$9syG&J&9!4m#;FwOsXvndG+OEIegQs(4qCqK8qCza*hd<F{<S(Zm6m=_`=+S#XX@EHkBUoh48D=XVyOeoWtgE~2?y!x;(R`Tu1{7d#%}fBU%<ng;q1xfML)f2dpwr*+Ajd!Zrk{jzf;%?NSonUYT6cBWq{)1LD#?smu>=Pl97Znx|d2S?1Qy}5FBV_C$Y6)Z>sAt_d715^igIqj;{y`1f_L@XC!QjSd8q9Gji2kh(}7@y8A`3JdJHyYu@%3@dj%BdDIQ&Cc>h;KR6r$lWQG-v<SE>~6ca6TC$i86B_a?T$k=}{|OTT^^9G{-Rdl(s2a=LC)ek=y|R@$p475ISPf(5HHXNNz(XJ1L`JGAqi4=}=E|t(fzo*J~N;XT45BX;Ps#IraAA$-Ez4rEFX|A~qq5W_eMH+~IO)>D&Vh=bi+Z0TXaKOO+P98Dtj2UAYEfFVvu}(;GP#M#+O0d}MgMnDEA|j^3^ba|{5Rg6$+TF{zPdQ%Kw~kSWR$9XuQM1rwIw3R?m{pGB5HY5zJmub4W%B9#yx7unUi&DdcMwD{V)&^-SvelD-AOs#5lUR3WWG#O!zxEaA}`LsslxYC+Fk<>8AOG`#6kVvpP1q>ZkIvosM`XDEt{9vA00gU)g*657XLv#iq_G?fvt*Kvv!geU+5<FJS`{59|-l&4Myg9W6oyKH^(pRIPGj~$;L90?3XE&0)g#RJvd8<_B+_QOMNwFK8`hmoyN)b(+T%hMEU=fl6V#hVAuEa`em)6$JMMP7|hOi6A9F7_nV5QH>RarA4o4tr$h1S@*G-^At6$8@eL|FhD3^;wpKnp+=^P3kVQvGS8r^8dg+PXrh)pFX=AARKi0y&M5`~',
 'diagnostics_snapshot.py': 'c-qBV-H+tPaeqJH{)hG&0f(nq?(M~Lg4qw?a(9By*2n3zN)T@rJPxO4hFT6eCONx1+6<6D4@MpW<d7JFVF=M#5ER)lVnqpp+&=hW75^Wbza+n^?k1ZY&g`CSBqwNx?2qc|`s(Vc;>O0tgG>g66oJSmK^%8QIvmDPBE=vIMoF3%Q7C@+_=y;1=>&tqEI5rtHcPr9O2T+Hh?0?*23ZjW+?;zG8ymN8MU!co6(Yz+=*#8pTRc5X<5-48lqR{*BOau)q>x!x4CFAF#l;{Bi*kCHpXiwbpjbo`X{MR?uAsjkrim;kjf+WKzC>yD$Ukw^_a?Fk03ay4KFHHVFEkB`F~*p6p5V0{HjRT~m}Zl*A)n{v^M}!N7{x$?{=;Sj$U5)|a))4c#c7ZwAXe`7Zr>94S7dYjME^2b%(8?r@!~WX<UGq`AcZ_DIx-1UBKDp$D~6kY?6|I)EYCtYEkx&?_a0?g3PS#A5YHq(yCTS`-&)2X;RVyFOa`5ybI$7|!9*^4;vAzFj=O}``J!cC@rsQ_Ly@EfQ__=XQC{R7cX_9WN2)1HPjj(XzMe+KSWQiV;PS3%OL+ktCKgCAMI^IHpX7*TH%f#=gUlc%9W}fwwrwmlqBPpWfN0uj_Ac|t{zqsVB3eu%c&@uVMRTd*%`$83f9u(FuJ+pGmnJM>R&fg|<oD-=%sbTH6~jozgOXK>G$4PJdhdY>$>{MrokX6}Huf9?_%w^4q8J5W-a{zppK(J;ScXZc<$DTz7h5Z`AP-pA_nZiZB$IV;9{-FXOn{_&h>t*%*hkpfa%;^&SLgg#&U;E*2D8aD@0<{2v2K#jGU*3-7)5(;1aU51Vsn&7NnQj=D9u1{DlcGlup8GE5(5SIwDY5sGA!Fw12vK!5Sl>1X%%!*tD?c8&J{%~ZzKzz-Eln@ZP`(Ry_rWNqK{!4tQwf%=d;Nq$mX4N23(W$bXnQtiA)MoAWJPVT5Jg?PDi;z4^eWGh5?&hk4HEb93u!Y;jD*5e~S+qK6Xc&1^OL_o1lq%d&1#)jtG)L)eptZcr`tSj|}IiW*pRd%U0ps=?8g~J3ZmdlHeqW;-DW(;Ok(Pq~;%tFaI4!PI&p1zy?JzzPk805~C<cM0j=ac|hQ0KTVIt<m%#Q(T+G)6Y|+yWPvEIE}n<t>*rS&Kc(kiiX08%G|i47Ue3az28f}sJlxzi$d`mPore~2{b*ozInB~hCi6CeuHloJwTej%+f@?0jsg?CYSI|NihKk1VpfB>H(3kNXgQ0!xeQZ<0Cf;S4;>i?(_9XG-DgNgs@OWS1Nj%5<;m~SaUJC;9mqk^(cB0M=g4)vA}xa0KsMF|r?<?Bt%lBo2O(Q>i_O5f7J@LHOk-Kd0cNPFiUMW3UyP&NuXYcH-8JeIj^$t$FU>96OPfuu*PgZ3R=Hy!Ry7k3Ovk>st1x|=-5y(eD!NsAq}KOY3oUH@Q+RtmQIfQ)&eopA)2;=RiHwbd$~|FG8~}z<66M$&I-q=UhxqKGy~*VsL}`W-O}}@=AG-Dm6>)>Yj|1Rsn1h*RC*W3;>{ZmZA<eQl_hE^A7}6P=4z(~-{EqnUU0Rw2pBhSf*0!r~WGDbvg4hO^Xx8-*;Ak%Gd=i{>uy`dRo7KkL67qHhnR#E1j&e4jv{L(;#!w#EmToN5nyIAcFGFoH<KO=J>f+Nv`}U)&m!Bsh8DG8ptk_ZJahzU$f;rS82ABvlnqB@PQ3g1=dik3~t>cMD2lqt?*ncD9%TENs{aL8tCnL|%cv{|Me-_1Hb{3AK6RFNQ{bEiyyA7&B9yyJJaPgxVtp<~g*sE*=UOVQ3;LVEAcp%!YT7KswNTQ)K50&Gt4YcWFyWDbi2E@JABUp8<vgh(_irWwyiE|iOoFzCwdZJ;X6^w;Z^f1jxZ8G_z8BkJN=$6}{dE_bb#FUh^V{p+^9oHMhX}{y#QQL9rzB!Gl(V*euS7Y@_uBJ5nSV(*UwUf2cYNS3!=<J~BbOsqR(4MU^q;jtK)?PW_Q1vQ0;RK-pM~#)Qx!^p0xb^<iH$^gnL`U!tUjA}cdI-hkuadDix%?!p3$^23w>i>Z_`1YL_Dii@;N3NzBW<=*heccIvazC{K{^Q{k}xu-IKXss5f#b@g2^prIX#VwV(M3~Kbi$`RLskk#v-PB7zZOBAC7~}59#VnZ|tmxMVj#o-A<eq%iXQn|K<|L{&OFMEjaaw?&dik2lrn8E^h{_m9G^Lcd<myLqm@v7RX94bhAqW(=HqJaV?Ow`e+)^0x@oz_98~X98xZ~9mXKSjP*d&kabW%9d%1)2wGdsZb~{HxgMMiiZH0VMr{={$Zh%smdP**p%-@Rw`X?xlzY001Pw(qod)i?!;XW^hOW4CM=1n@Z`TJS`$}$t0@tZ&(IVnXF5N|SuTtA0D@RcG;$YGr1Y|YYgenx=hA#*2Ca;;>x@*oLwItH6D7dxkG;4#eoMsJ1+p^k1#wJ7SMr`Gm?*hDF&HdIpX2WMR6JCx`v(%V|x~s84NFfFcUAio^O;x7At@T-==#rno_h-F{{@hpUM8|6ts_W?Z-SP!q5y=vo62f7*U}f(d7xp}tG3zqO`P!b?UXBSUa`kL2nyK6Gb-<fb4l7Tywwg5pH}=kIiUTX$P><nYz`)oL^p%~2%yP;`GT~s9_MH<79F%t#@{<ud_34a4CoBLTSGH3Z4SY7fJ_lWMI6N3w`X%~ET^iypfk>y|@LCA95pLZb*6iuAol^`Z+$&P{EDOl)zg8mMq2F3PesgiqVA<>CFru2%xQ2iU%IuO~Z3|+j#<q5LwaAoP!>P{gn=G_mKCWHRvUN9E%{Gl{sRSnf6*i`{zp(^0wpk}?;yyCh>kJ8<aJA*M<%4R52*#>iksRpu+F0D>a|(Hp5u#&`#zL1mY+`GNq_IqGyR{*fHUW^%)sx<_BC#Tu>gXC%Qu)p0bQ9up7dCVip#$jaLv^Iy<WHLhOfru(u?`VAqnq1%2|TYcWFI%FmtT+@nqIyDirP<|qV>~A1jxN0+Y#hrToX^X$gjfNT%29L2>JT<tAqjI&SoxPB7r{}iNUOX7h^NaK}&K#{!d)eZ|aoR{Wfl>IbqAoT;s;Jdahd3ZtTapCh8}Pf-Jukm)2s@7MZ3D+5lRycTdk(l0>0JHR;d@nLfVK6PcYvl5V(%mfY-q3;I>er;<}&`@8fPQ70r#Hcq0A{as3T2_kMTi@gmFH=~s~xc|VGFckpU-BMjPEuG!Gn4_(q4(8`W+<$u8uTRe?{{_|9oJBi8cLepi_nqn44kZsZ$1)m?G0b~?QBVl?ybt}9*(Rj`i6K4nuyYy>ig9oI?%h9F;HFs&^vK>eEs)01Kzw^J80<`g0i6TAKf($``%OsCDCxy=SnPl<NK*!}Zo?V2030IMFiuZ5PqSd!3zGTiSY~noE}fr_q0^f=9AVkRJC9gss-K6m3?`sAO(Qaki`^}?cbgtVXLr2aMm+I9fAtUIoH*dgJPf8B(%{qF1t836K$-3vW$)FeH9Q1&;fe2AQJ8c4oLy|4TL<!D2f^`wW8J^_SPeAi?!w#Mnl>0a-G5}9Ope)n{rvJ%wi_?zqSV11VZ2Ko1JxDB(bdIA69j@drv9b6E5ojEvY4Xt2*3ZdQ1K(aXKNe11v2sAkKjl4XQR@r5x!pl81J%AQ7a6e&hl2Azy7<+pU)YW+!#v`=0#+JTwg!e=%Fzsly)d^^{>eHU0wWm#`w|5kJL?nN?Z6WL=)^M;r5dVSVfAZ$b5bMJWvRNC@tuW@9o>cw$<S@NWi`QT?pL1BVH3WyDf{Mbp9HH;bOi6OfuPmOUBspx@U4CVkXQMelPx!erfXv{C&*W)MvA*!Zu|1f4u+rA#A~z9}hABp%}^d@<mj(zWbnjd_cmc*C-niYEjQyfM`}aTJr0)J7D?soFg;&3UirU^6I*)Z$#a;Jf0)|QaIiR(Ch2m7*Mz4><HqLcDxSt8Fb{<lO^g_s_xy6MoCAetV@Tn+~%_E5Qz!8qw#?$C<gmA3qT92Oe1jG*l@{!V2s0*D^rk)Qd%4cs9QfPkby5~#Y5^=+#TRNh~j*IR~x+j18Bl?F!%WK7vlTxz4JE7@ypMZCVUR_r-v!Q?bhU09eL#8C}G^)Dxq)jX;fap$pe~DlyvAg&bLL2?Sut1#*zcXZ(UvdlXzBcgup$+wQ|O)If~c`N-n=hv^617hZ_F`8v#=j@Aj=t@l4~+q0VQ$T3ipMBeqa?#k&u>oDxKfaSE+$;TDIsuZfqdMNMb|>2j=Ts!jA{n8m-15Qo%;mUM9}!(%>emTV!Ir$A~I(I8pNC<aV~*~;Vxd@q9!ORirZgh|AR%$HVl#P6?O{wJ9?+Ww5(MFmx#3mU49jfUgj{yJzNCWsZQOsm<njt41VjlsH?a4yIj`rg686QPuqBCgstahl@sLjzF9&NReuJmW$5odaef#++VY{#nyxauQ`}Le~w$^(XUUj6{|IN<UT@{q;<Yam;YE7yfia$qYre0#L;Xof2AtC`rEsAmuru4~4FBz^T-M>wgGRRWg<9Gh=Z1U!boNMLwQ$f<<CQv<bxPE$rG-#2|onH$>w_2no6T=LvBTzC7^~lXh726?ngvtI&i@`)Scwcl$0u{?dpVe$^<iIK!4Ei~!xj>?z`{=uZ?xBhm!ZEZC(J=QE<~zi7JlFCibH`tubWftUZ0>@*tH865ui7hopLKuKfLa70Av2QM#8$#PnjQcoep@CTBl7{wHkL|P;!#*K~tLRi=;FFK9wh;YUYYQuo(dSrRnh2bogR>a?%FC(5}h62I+SN19yUH%-q2Y<)J86dWE`<B9q?XgbFsCi9}K;-LJoM$j>DRHziL$`1BgA|tP;_oNKrcbqV2yDL!wm8Wc!|Q--lelW@SLQ`nhGA*HQvOk`f`T_#vxlA{UquLFgmOVj&C#8<3t3UBexoH8n*0o{UfCopWO>BB8@t54rcZqP7EUB}Na<vvhoVKYACqI4ppJUU_YN|yFrvGmpC0q8xwuKH0aQ^x1A+?KIHG4Vlzgz!IqembsX3-CYef-5XUe9ELLc+o!^xLZ1|DHFU3rKO^XQoe4j<wsa~zB~XmY+-*ttM%SCRJVJb=4$YTcvxjtlxj)nSUWfL^Eb$!t0pa!-sjo#mD-m%~~3A67^l9qLEQXe`%VnvHezC@UAC>*QMIJW*rcPK!4vjm|}MuZ~QW8maGv%MNu3r}x8X`UWLhO16R&L^TRkQYiw}w3TBYKk*+vdgIOe2ag_dabh3H)#Bb9`{qVfzD9nX^YqdEhi^UdCKPH>o>9FX%xM-8E<vdIMZ~PXNVG3$dfZn9X*)}RsC>Iu1!7qpB3V^_XfoHj0`WRHmd8_rquXx`$MSJ%a7<izMZn9w!A2{Gr?FGju>|^tDX!+qr6XjOE0np4ahglY$!M}1_Fiutu{Q6ehAdhkzjj^Ls*(RHS3COsp`xt#pk@0MlIza~xQd|@C+8jfx%VCY(<t`kYIU5FbY>&OhTB_%&TBJGYbqof^%Zy?ZdAKlusUt5E#|9SU42_K$<*rdd*=@4P#5d6wX1T?^_sy|3|mo(yj<;2Ck2N`L5~|zVHrpm<dRRm=7!BJUpMNvqvgo59?)n#>aMQj)kykABmhFkS2)JoY+kfevf8~zYoekTenB36D8=rcEru)TYTJ{J1q_C5!(0_KF}IfJ=&s-uh<UiQJ39BsS!t<Cuh&|x1eG60Qy&6P=K?`9?Q~j86v%D+a73N;MXFVi!m5gT)=N|WV6r}@_yOl`8lysN#m>1_jv}T&6Rr&0Ex$;s(x>3mu|TNgq8_hGDtfD>#Q-Fo#}perg)1tkUVu`;G>^_!Az4a}1_<@Dt<oACERNU$x|*{Uhc#uEMYOV=*DPF@)Znyh>e})x3SpV;xa*-<rh0R)1*P?~Z8~|#1axk=>Nh0XO0`&qyQ~}KMzwylm1Nvxoi%HiWNDiDkik3}NY0z810c=jEeq;Hv@8Px@asJsD+1GSadc1R<4`#^-x*RXEqJ{b)fTn@06T-G5WglM{;F<jGfY%p)9U@**5xmg=-$T{Fns~`O`OA<&e<cRKvJ_&g_WAER)ne6)n>qI2QwkjYM}W5yUq^QOn^G291e_;0pRS`^0rr%p>mvQxeE0VDZr!NY7~FxGghOmqhW-2Ss1#OX-&~oTtvP(8AhYBF;8b%D5+YVuZ5=BcqN%6ZtCVWrp0;p{rhh|KKOJ0>7ys_JUvhsL?a;e>zQ-^{fCba>I2QUFHL<~w_-Vm?k&)2F4Uknpe2QMaQ_d&Rz@6OUEuihno`{A_0F$~TI1_PC+n~HB5}$3s?Wc2DUZzu0BuseX$!Tx=4@NCXB-!;33b;jN;j2`4Xli7pbJr?P&SI_xzZ!H!z0y<S%UOF_FxX6S{%%L>!t*?D)OfrbxBkHAUNYlnff@`yPu7?Jm?AcbOy*R;e$;6bkL{TA>ViHsdWD1ivcyK<7{q%(T=K!hxiU`gjP(@+g&jUBk6*w=kgEDF25tHD*WmnFVGlIf2}|T=rx*7@o{Bgr+LpEgGH}4zEMwCmV#LIinY4Ba)4<%TDbZ{LP)jGAxPinuep4m7~%U)&wTShY`J~we*v7tHN5',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC'}
EMBEDDED_SHA256 = {'audit_snapshot.py': '4d08e3ea1613e4248d35ebfb40716a5fafbf1a6916bfe35a99a74686d8093bc7',
 'crawler_snapshot.py': '4f3e6e74821404f0215be3157a7259b5012e827e54fca31921ee4ec078612ed7',
 'diagnostics_snapshot.py': 'f03a5b21ef02173598b2571b69388f1d43a0695e676fb63340be70238b7b63d4',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if not target.exists():
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
# Diagnostics chỉ đọc output, có thể cập nhật riêng mà không đổi crawler/hash resume.
diagnostic_body = zlib.decompress(base64.b85decode(EMBEDDED_FILES['diagnostics_snapshot.py']))
if hashlib.sha256(diagnostic_body).hexdigest() != EMBEDDED_SHA256['diagnostics_snapshot.py']:
    raise ValueError('Snapshot diagnostics bị hỏng.')
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
DIAGNOSTICS.write_bytes(diagnostic_body)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        raise
    finally:
        refresh_diagnostics()
if (RUN_DIR / 'crawl_config.json').exists():
    saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
    if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
        raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
if not requirements_path.exists():
    requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script == CRAWLER else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Crawler tuân thủ robots,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File có một số mẫu record/raw để debug, không nén lại toàn bộ raw.


In [ ]:
with diagnostic_step('crawl'):
    arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS,
                 '--delay', DELAY, '--attempts', ATTEMPTS]
    if MAX_NEW is not None:
        arguments += ['--max-new', MAX_NEW]
    print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
    run_script(CRAWLER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Corpus accepted chỉ gồm tài liệu qua heuristic; chưa chứng minh chất lượng bằng chấm thủ công.
Audit chỉ chạy khi đủ checkpoint. Nếu đang crawl thử, vẫn xuất phần accepted hiện có;
manifest notebook ghi rõ đây là corpus chưa hoàn tất.


In [ ]:
with diagnostic_step('report_audit_export'):
    run_script(CRAWLER, 'report', '--out', RUN_DIR)
    summary = json.loads((RUN_DIR / 'quality_report.json').read_text(encoding='utf-8'))
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if summary['pending'] == 0:
        run_script(AUDITOR, '--out', RUN_DIR)
    else:
        print('Run chưa hoàn tất; bỏ qua audit toàn mẫu. Chạy lại ô crawl để tiếp tục.')
    if summary['accepted'] > 0:
        run_script(EXPORTER, '--out', RUN_DIR)
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Chưa có tài liệu accepted; xem quarantine.jsonl và QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
from IPython.display import FileLink
import zipfile

refresh_diagnostics(strict=True)
with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
    diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
print('ZIP chẩn đoán trong Output:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint.


In [ ]:
from IPython.display import FileLink

notebook_manifest = {
    'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
    'dataset_revision': DATASET_REVISION, 'source_sha256': SOURCE_SHA256,
    'selected': summary['selected'], 'completed': summary['completed'],
    'pending': summary['pending'], 'accepted': summary['accepted'],
    'complete': summary['pending'] == 0,
    'packages': versions,
    'exported_files': [name for name in ['corpus.parquet', 'corpus.jsonl'] if (RUN_DIR / name).exists()],
}
(RUN_DIR / 'notebook_manifest.json').write_text(json.dumps(notebook_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
refresh_diagnostics(strict=True)
archive = shutil.make_archive(str(WORKDIR / RUN_DIR.name), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('ZIP:', archive)
print('Corpus:', RUN_DIR / 'corpus.parquet')
display(FileLink(str(Path(archive).relative_to(WORKDIR))))
print('ZIP chẩn đoán:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
if (RUN_DIR / 'corpus.parquet').exists():
    display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
    display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))
